<a href="https://colab.research.google.com/github/PeterMaZep/SyntheticMcWikiGenerator/blob/main/Meta_Synthetic_Data_Llama3_2_(3B).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!

Custom version of Synthetic Data Generator for Minecraft Wiki!

### News

March update 2026 in progress!

### Installation

In [ ]:
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm synthetic-data-kit==0.0.3
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install synthetic-data-kit==0.0.3
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

## Primary Goal
Our goal is to make Llama 3.2 3B understand the "Byte Latent Transformer: Patches Scale Better Than Tokens" [research paper](https://ai.meta.com/research/publications/byte-latent-transformer-patches-scale-better-than-tokens/) that was published in December 2024.

We'll use https://github.com/meta-llama/synthetic-data-kit to generate question and answer pairs **fully locally** which will be used for finetuning Llama 3.2 3B!

In [ ]:
from unsloth.dataprep import SyntheticDataKit

generator = SyntheticDataKit.from_pretrained(
    # Choose any model from https://huggingface.co/unsloth
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 2048, # Longer sequence lengths will be slower!
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 05-01 14:15:23 [__init__.py:239] Automatically detected platform cuda.


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Using dtype = torch.float16 for vLLM.
Unsloth: vLLM loading unsloth/Llama-3.2-3B-Instruct with actual GPU utilization = 89.39%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 192.
Unsloth: vLLM's KV Cache can use up to 7.19 GB. Also swap space = 2 GB.
vLLM STDOUT: INFO 05-01 14:15:38 [__init__.py:239] Automatically detected platform cuda.
vLLM STDOUT: INFO 05-01 14:15:39 [api_server.py:981] vLLM API server version 0.8.2
vLLM STDOUT: INFO 05-01 14:15:39 [api_server.py:982] args: Namespace(subparser='serve', model_tag='unsloth/Llama-3.2-3B-Instruct', config='', host=None, port=8000, uvicorn_log_level='info', disable_uvicorn_access_log=False, allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key=None, lora_modules=None, prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', 

## Generate QA Pairs + Auto clean data
We now use synthetic data kit for question answer pair generation:

In [ ]:
generator.prepare_qa_generation(
    output_folder = "data", # Output location of synthetic data
    temperature = 0.7, # Higher temp makes more diverse datasets
    top_p = 0.95,
    overlap = 64, # Overlap portion during chunking
    max_generation_tokens = 512, # Can increase for longer QA pairs
)

Check if it succeeded:

In [ ]:
!synthetic-data-kit system-check

 VLLM server is running at http://localhost:8000/v1
Available models: {'object': 'list', 'data': [{'id': 
'unsloth/Llama-3.2-3B-Instruct', 'object': 'model', 'created': 1746109670, 
'owned_by': 'vllm', 'root': 'unsloth/Llama-3.2-3B-Instruct', 'parent': None, 
'max_model_len': 2048, 'permission': [{'id': 
'modelperm-4b22dda9834e436cb033f1d00ee42ee8', 'object': 'model_permission', 
'created': 1746109670, 'allow_create_engine': False, 'allow_sampling': True, 
'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 
'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': 
False}]}]}
⠋ Checking VLLM server at http://localhost:8000/v1...


## Document Parsing (PDF, CSV, HTML etc.)
Now, let's take the Byte Latent Transformer: Patches Scale Better Than Tokens research paper found at https://arxiv.org/abs/2412.09871 and covert it to Q&A pairs in order to finetune Llama 3.2!

## Loading previous data


In [ ]:
from datasets import load_dataset
import json, os, glob

HF_REPO = "PWindows/minecraft-wiki-qa"
from google.colab import userdata
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✓ Token loaded from Colab secrets")
except:
    import getpass
    HF_TOKEN = getpass.getpass("Enter HuggingFace token: ")

try:
    existing = load_dataset(HF_REPO, token=HF_TOKEN)
    completed_pages = set(existing["train"]["source"])
    print(f"✓ Resuming — {len(completed_pages)} pages already done")
except:
    completed_pages = set()
    print("✓ Starting fresh")

## Fetching wiki


In [ ]:
import requests, time

WIKI_API = "https://minecraft.wiki/api.php"

def get_all_titles():
    titles = []
    params = {"action":"query","list":"allpages","aplimit":"500","format":"json","apnamespace":"0"}
    while True:
        r = requests.get(WIKI_API, params=params, timeout=30).json()
        titles.extend([p["title"] for p in r["query"]["allpages"]])
        if "continue" not in r: break
        params["apcontinue"] = r["continue"]["apcontinue"]
        time.sleep(0.5)
    return titles

def get_page_text(title):
    params = {"action":"query","titles":title,"prop":"extracts","explaintext":True,"format":"json"}
    r = requests.get(WIKI_API, params=params, timeout=30).json()
    page = next(iter(r["query"]["pages"].values()))
    text = page.get("extract", "")
    return text[:3000] if len(text) > 200 else None

titles = get_all_titles()
print(f"✓ Found {len(titles)} pages")

# QA Generation

In [ ]:
import time, json, glob, os
from datasets import Dataset

os.makedirs("data/input", exist_ok=True)
BATCH_SIZE = 50
batch = []

for i, title in enumerate(titles):
    if title in completed_pages:
        continue

    print(f"[{i}/{len(titles)}] {title}")

    text = get_page_text(title)
    if not text:
        print("  Skipping (no content)")
        completed_pages.add(title)
        continue

    # Save to temp file
    with open("data/input/page.txt", "w") as f:
        f.write(text)

    try:
        # Chunk and generate
        chunks = generator.chunk_data("data/input/page.txt")
        for chunk in chunks[:1]:
            !synthetic-data-kit \
                -c synthetic_data_kit_config.yaml \
                create {chunk} \
                --num-pairs 7 \
                --type "qa"
            time.sleep(2)

        # Load generated pairs
        for fn in glob.glob("data/generated/page_*_qa_pairs.json"):
            with open(fn) as f:
                pairs = json.load(f)
            for p in pairs:
                batch.append({
                    "question": p["question"],
                    "answer": p["answer"],
                    "source": title
                })
            os.remove(fn)  # cleanup

    except Exception as e:
        print(f"  Failed: {e}")
        continue

    completed_pages.add(title)

    # Push every BATCH_SIZE pages
    if len(batch) >= BATCH_SIZE:
        Dataset.from_list(batch).push_to_hub(HF_REPO, token=HF_TOKEN)
        print(f"✓ Pushed {len(batch)} pairs")
        batch.clear()

# Final push
if batch:
    Dataset.from_list(batch).push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"✓ Final push: {len(batch)} pairs")

⠴ Processing https://arxiv.org/html/2412.09871v1...
 Text successfully extracted to data/output/arxiv_org.txt
37 ['data/output/arxiv_org_0.txt', 'data/output/arxiv_org_1.txt', 'data/output/arxiv_org_2.txt']


Let's load up the data and see what the synthetic data looks like!

In [ ]:
from datasets import Dataset
import pandas as pd
final_filenames = [
    f"data/final/arxiv_org_{i}_qa_pairs_ft.json"
    for i in range(len(filenames[:3]))
]
conversations = pd.concat([
    pd.read_json(name) for name in final_filenames
]).reset_index(drop = True)

dataset = Dataset.from_pandas(conversations)

In [ ]:
dataset[0]

{'messages': [{'content': 'You are a helpful assistant.', 'role': 'system'},
  {'content': 'What is the primary unit of computation in the Byte Latent Transformer (BLT) architecture?',
   'role': 'user'},
  {'content': 'Patches', 'role': 'assistant'}]}

In [ ]:
dataset[1]

{'messages': [{'content': 'You are a helpful assistant.', 'role': 'system'},
  {'content': 'How do patches in BLT architecture get segmented?',
   'role': 'user'},
  {'content': 'Based on the entropy of the next byte', 'role': 'assistant'}]}

In [ ]:
dataset[-1]

{'messages': [{'content': 'You are a helpful assistant.', 'role': 'system'},
  {'content': 'What is the main purpose of the formal definition of the patching function?',
   'role': 'user'},
  {'content': 'to formally describe the process of segmenting bytes into patches',
   'role': 'assistant'}]}

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>